# Matplotlib 3/3 (option avancée) : histogrammes, boîtes à moustaches et plus

**Objectifs** : représenter la **distribution** d'une variable (histogramme, boxplot, violon), visualiser une matrice (heatmap) et utiliser le raccourci `.plot` de pandas.

*Notebook optionnel. Durée estimée : 40 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/cdrutinuspro/intro-data-python.git
    %cd intro-data-python/notebooks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/titanic.csv")

## 1. Histogramme : `hist`

Un histogramme découpe l'intervalle des valeurs en **classes** (*bins*) et compte combien de valeurs tombent dans chacune. Il répond à la question : « comment sont réparties mes valeurs ? »

In [ ]:
ages = df["Age"].dropna()

fig, ax = plt.subplots()
ax.hist(ages, bins=20, edgecolor="black")
ax.set_xlabel("Âge")
ax.set_ylabel("Nombre de passagers")
ax.set_title("Distribution des âges")
plt.show()

Le paramètre **`bins`** change beaucoup l'allure : trop peu masque la forme, trop crée du bruit. Comparez :

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=False)
for ax, b in zip(axes, [5, 20, 80]):
    ax.hist(ages, bins=b, edgecolor="black")
    ax.set_title(f"bins = {b}")
plt.show()

### Comparer deux distributions

Pour comparer des groupes de tailles différentes, on utilise `density=True` : l'aire de l'histogramme vaut 1, les groupes deviennent comparables. `alpha` permet de superposer.

In [ ]:
a_surv = df[df["Survived"] == 1]["Age"].dropna()
a_mort = df[df["Survived"] == 0]["Age"].dropna()

fig, ax = plt.subplots()
ax.hist(a_mort, bins=20, alpha=0.5, density=True, label="Décédés")
ax.hist(a_surv, bins=20, alpha=0.5, density=True, label="Survivants")
ax.axvline(a_surv.mean(), color="tab:orange", linestyle="--")
ax.axvline(a_mort.mean(), color="tab:blue", linestyle="--")
ax.set_xlabel("Âge")
ax.set_ylabel("Densité")
ax.legend()
plt.show()

---
### ✏️ Exercices (histogrammes)

### ✏️ Exercice 1

Tracez l'histogramme des tarifs (`Fare`) en 30 classes. Ajoutez deux lignes verticales : la moyenne (rouge) et la médiane (verte). Que remarquez-vous sur l'écart entre les deux ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
fig, ax = plt.subplots()
ax.hist(df["Fare"], bins=30, edgecolor="black")
ax.axvline(df["Fare"].mean(), color="red", label="moyenne")
ax.axvline(df["Fare"].median(), color="green", label="médiane")
ax.legend()
plt.show()
# La distribution est très asymétrique : quelques tarifs très élevés tirent la moyenne vers la droite.
```

</details>

## 2. Boîte à moustaches : `boxplot`

Une boîte à moustaches résume une distribution en 5 nombres :

- la **boîte** va du 1er quartile (Q1) au 3ème quartile (Q3) : elle contient 50 % des données
- le **trait central** est la médiane
- les **moustaches** s'étendent jusqu'aux valeurs les plus éloignées situées à moins de 1,5 × (Q3 − Q1) de la boîte
- les **points isolés** au-delà sont des valeurs atypiques (*outliers*)

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(ages)
ax.set_ylabel("Âge")
plt.show()

Pour comparer plusieurs groupes, on passe une **liste** d'arrays :

In [ ]:
groupes = [df[df["Pclass"] == c]["Age"].dropna() for c in [1, 2, 3]]

fig, ax = plt.subplots()
ax.boxplot(groupes, tick_labels=["1ère", "2ème", "3ème"])
ax.set_xlabel("Classe")
ax.set_ylabel("Âge")
ax.set_title("Âge selon la classe")
plt.show()

> Remarque : le paramètre `tick_labels` existe depuis Matplotlib 3.9. Sur une version plus ancienne, on utilisait `labels=`.

### Violin plot

Un *violin plot* ajoute la **forme** de la distribution (densité) à droite et à gauche d'un axe central.

In [ ]:
fig, ax = plt.subplots()
ax.violinplot(groupes, showmedians=True)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["1ère", "2ème", "3ème"])
ax.set_ylabel("Âge")
plt.show()

---
### ✏️ Exercices (boxplot)

### ✏️ Exercice 2

Tracez un boxplot du tarif (`Fare`) pour chacune des trois classes. Les tarifs de 1ère classe écrasent l'échelle : ajoutez `ax.set_yscale("log")`... mais attention, certains tarifs valent 0. Filtrez-les d'abord.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
d = df[df["Fare"] > 0]
donnees = [d[d["Pclass"] == c]["Fare"] for c in [1, 2, 3]]
fig, ax = plt.subplots()
ax.boxplot(donnees, tick_labels=["1ère", "2ème", "3ème"])
ax.set_yscale("log")
ax.set_ylabel("Tarif (échelle log)")
plt.show()
```

</details>

### ✏️ Exercice 3

Comparez l'âge des hommes et des femmes avec un boxplot (deux boîtes).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
h = df[df["Sex"] == "male"]["Age"].dropna()
f = df[df["Sex"] == "female"]["Age"].dropna()
fig, ax = plt.subplots()
ax.boxplot([h, f], tick_labels=["Hommes", "Femmes"])
ax.set_ylabel("Âge")
plt.show()
```

</details>

## 3. Visualiser une matrice : `imshow` (heatmap)

`imshow` affiche une matrice sous forme d'image, chaque case étant colorée selon sa valeur. Idéal pour une **matrice de corrélation** ou un tableau croisé.

In [ ]:
cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
corr = df[cols].corr()
corr.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label="Corrélation")

ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticklabels(cols)

# Afficher la valeur dans chaque case
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Matrice de corrélation")
plt.show()

`vmin=-1, vmax=1` fixe l'échelle de couleurs, de sorte que 0 (aucune corrélation) soit toujours au centre de la palette `coolwarm`.

## 4. Le raccourci `.plot` de pandas

Les Series et DataFrames ont leurs propres méthodes de tracé, qui utilisent Matplotlib en coulisses et **renvoient un `ax`** : on peut donc les personnaliser ensuite avec tout ce que vous connaissez déjà.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df["Age"].plot.hist(bins=20, ax=axes[0], title="Histogramme")
df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[1], title="Survie par classe")
df.plot.scatter(x="Age", y="Fare", alpha=0.4, ax=axes[2], title="Âge vs tarif")

plt.tight_layout()
plt.show()

In [ ]:
# boxplot par groupe en une ligne
df.boxplot(column="Age", by="Pclass")
plt.suptitle("")
plt.show()

**Quand utiliser quoi ?** `.plot` est pratique pour explorer vite. Matplotlib « pur » donne un contrôle total pour les figures finales.

## Récapitulatif

| Graphique | Question à laquelle il répond | Code |
|---|---|---|
| histogramme | comment sont réparties les valeurs ? | `ax.hist(x, bins=..., density=...)` |
| boxplot | médiane, dispersion, valeurs atypiques, par groupe | `ax.boxplot([g1, g2, ...])` |
| violin | idem, avec la forme de la distribution | `ax.violinplot([...])` |
| heatmap | valeurs d'une matrice | `ax.imshow(M, cmap=..., vmin=..., vmax=...)` |
| raccourcis pandas | exploration rapide | `df["col"].plot.hist()`, `df.plot.scatter(...)` |

---
## Exercices (pour finir)

### ✏️ Exercice 4

Construisez avec `pivot_table` le taux de survie par classe (lignes) et sexe (colonnes), puis affichez-le avec `imshow` (cmap `"RdYlGn"`, valeurs écrites dans les cases).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
t = df.pivot_table(index="Pclass", columns="Sex", values="Survived", aggfunc="mean")
fig, ax = plt.subplots()
im = ax.imshow(t, cmap="RdYlGn", vmin=0, vmax=1)
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(t.columns)))
ax.set_xticklabels(t.columns)
ax.set_yticks(range(len(t.index)))
ax.set_yticklabels(t.index)
ax.set_ylabel("Classe")
for i in range(t.shape[0]):
    for j in range(t.shape[1]):
        ax.text(j, i, f"{t.iloc[i, j]:.2f}", ha="center", va="center")
plt.show()
```

</details>

### ✏️ Exercice 5

**Défi** : construisez une figure 1×3 résumant le Titanic : (1) histogramme de l'âge, (2) boxplot du tarif par classe (échelle log), (3) barres du nombre de passagers par port. Ajoutez des titres et un titre général.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axes[0].hist(df["Age"].dropna(), bins=20, edgecolor="black"); axes[0].set_title("Âge")
d = df[df["Fare"] > 0]
axes[1].boxplot([d[d["Pclass"] == c]["Fare"] for c in [1, 2, 3]], tick_labels=["1", "2", "3"]); axes[1].set_yscale("log"); axes[1].set_title("Tarif par classe")
p = df["Embarked"].value_counts()
axes[2].bar(p.index, p.values); axes[2].set_title("Passagers par port")
fig.suptitle("Le Titanic en un coup d'œil")
plt.show()
```

</details>